# 11 LangGraph Workflow

This notebook connects the main components of the CV-job matching system into one workflow using LangGraph.

The goal of this notebook is to demonstrate how the system can work as an AI workflow instead of a set of isolated notebooks.

In this notebook, the workflow uses output files generated by previous notebooks.

This notebook demonstrates the orchestration logic of the system.

## 1. Imports and Settings

In [117]:
import json

from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, TypedDict
import pdfplumber

from langgraph.graph import StateGraph, START, END

In [88]:
CV_PDF_PATH = Path("../data/raw/test_cv/CV_searchable_ATS.pdf")
CV_TEXT_OUTPUT_PATH = Path("../data/processed/test_cv/test_cv_extracted_text.txt")

CV_QUALITY_DIR = Path("../outputs/cv_quality")
CV_EXTRACTION_DIR = Path("../outputs/cv_extraction")
JOB_EXTRACTION_DIR = Path("../outputs/job_extraction")
MATCHING_DIR = Path("../outputs/matching")
RECOMMENDATIONS_DIR = Path("../outputs/recommendations")
FINAL_REPORT_DIR = Path("../outputs/final_report")
LANGGRAPH_OUTPUT_DIR = Path("../outputs/langgraph_workflow")

In [89]:
cv_quality_path = CV_QUALITY_DIR / "cv_quality_analysis.json"
structured_cv_path = CV_EXTRACTION_DIR / "structured_cv.json"
structured_job_path = JOB_EXTRACTION_DIR / "structured_job.json"
matching_result_path = MATCHING_DIR / "matching_result.json"
recommendations_path = RECOMMENDATIONS_DIR / "recommendations.json"

final_report_json_path = FINAL_REPORT_DIR / "final_report.json"
final_report_markdown_path = FINAL_REPORT_DIR / "final_report.md"

workflow_result_path = LANGGRAPH_OUTPUT_DIR / "workflow_result.json"
workflow_log_path = LANGGRAPH_OUTPUT_DIR / "workflow_log.md"

## 2. Define Workflow State

In [90]:
class CVJobMatchingWorkflowState(TypedDict, total=False):
    cv_pdf_path: str
    cv_text_output_path: str
    cv_quality_path: str
    structured_cv_path: str
    structured_job_path: str
    matching_result_path: str
    recommendations_path: str
    final_report_json_path: str
    final_report_markdown_path: str

    cv_text: str
    cv_quality_analysis: Dict[str, Any]
    structured_cv: Dict[str, Any]
    structured_job: Dict[str, Any]
    matching_result: Dict[str, Any]
    recommendations_output: Dict[str, Any]

    final_report: Dict[str, Any]
    final_markdown_report: str

    started_at: str
    completed_at: str
    status: str
    workflow_log: List[str]
    errors: List[str]

    workflow_output: Dict[str, Any]

## 3. Define Helper Functions

In [ ]:
def load_json_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

In [ ]:
def save_json_file(data, file_path):
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

In [ ]:
def append_workflow_log(state, message):
    current_log = state.get("workflow_log", [])
    timestamp = datetime.now().isoformat(timespec="seconds")
    return current_log + [f"{timestamp} | {message}"]

In [ ]:
def create_markdown_list(items, empty_message="- No items available."):
    if not items:
        return empty_message

    lines = []

    for item in items:
        if item is None:
            continue

        if isinstance(item, dict):
            item_text = (
                item.get("job_requirement")
                or item.get("title")
                or item.get("skill")
                or str(item)
            )
        else:
            item_text = str(item)

        if item_text.strip():
            lines.append(f"- {item_text}")

    if not lines:
        return empty_message

    return "\n".join(lines)

In [ ]:
def create_matched_items_list(items, empty_message="- No matched items available."):
    if not items:
        return empty_message

    lines = []

    for item in items:
        if not isinstance(item, dict):
            lines.append(f"- {item}")
            continue

        job_requirement = item.get("job_requirement")
        cv_evidence = item.get("cv_evidence")
        match_type = item.get("match_type")

        line = f"- {job_requirement}"

        if cv_evidence:
            line += f" | CV evidence: {cv_evidence}"

        if match_type:
            line += f" | Match type: {match_type}"

        lines.append(line)

    return "\n".join(lines)

In [ ]:
def get_recommendations_content(recommendations_output):
    if not isinstance(recommendations_output, dict):
        return {}

    return recommendations_output.get("recommendations", recommendations_output)

In [ ]:
def extract_text_from_pdf(pdf_path):
    extracted_pages = []

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()

            if page_text:
                extracted_pages.append(page_text)

    extracted_text = "\n\n".join(extracted_pages).strip()

    return extracted_text

## 4. Define Compact Final Report Helper

In [ ]:
def create_compact_final_report_from_state(state):
    structured_cv = state.get("structured_cv", {})
    structured_job = state.get("structured_job", {})
    cv_quality_analysis = state.get("cv_quality_analysis", {})
    matching_result = state.get("matching_result", {})
    recommendations_output = state.get("recommendations_output", {})

    recommendations = get_recommendations_content(recommendations_output)

    final_result = matching_result.get("final_result", {})
    score_breakdown = matching_result.get("score_breakdown", {})
    matched_items = matching_result.get("matched_items", {})
    missing_items = matching_result.get("missing_items", {})
    semantic_analysis = matching_result.get("semantic_analysis", {})

    compact_report = {
        "metadata": {
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "report_type": "compact_final_report_generated_inside_langgraph_workflow",
            "note": "This compact report was generated because notebook 10 final_report.json was not found."
        },
        "candidate_information": {
            "candidate_name": structured_cv.get("candidate_name"),
            "email": structured_cv.get("email"),
            "phone": structured_cv.get("phone"),
            "location": structured_cv.get("location")
        },
        "job_information": {
            "job_title": structured_job.get("job_title"),
            "company_name": structured_job.get("company_name"),
            "job_category": structured_job.get("job_category"),
            "location": structured_job.get("location"),
            "work_mode": structured_job.get("work_mode"),
            "employment_type": structured_job.get("employment_type")
        },
        "cv_quality_summary": {
            "overall_summary": cv_quality_analysis.get("overall_summary"),
            "scores": cv_quality_analysis.get("scores", {}),
            "strengths": cv_quality_analysis.get("strengths", []),
            "weaknesses": cv_quality_analysis.get("weaknesses", [])
        },
        "matching_summary": {
            "final_hybrid_score": final_result.get("final_hybrid_score"),
            "rule_based_score": final_result.get("rule_based_score"),
            "semantic_score": final_result.get("semantic_score"),
            "match_category": final_result.get("match_category"),
            "score_breakdown": score_breakdown
        },
        "skills_summary": {
            "matched_required_skills": matched_items.get("matched_required_skills", []),
            "missing_required_skills": missing_items.get("missing_required_skills", []),
            "matched_technology_skills": matched_items.get("matched_technology_skills", []),
            "missing_technology_skills": missing_items.get("missing_technology_skills", [])
        },
        "semantic_analysis": semantic_analysis,
        "recommendation_summary": {
            "overall_recommendation_summary": recommendations.get("overall_recommendation_summary"),
            "priority_actions": recommendations.get("priority_actions", []),
            "evidence_based_notes": recommendations.get("evidence_based_notes", [])
        }
    }

    return compact_report

In [ ]:
def create_compact_markdown_report(final_report):
    candidate_information = final_report.get("candidate_information", {})
    job_information = final_report.get("job_information", {})
    cv_quality_summary = final_report.get("cv_quality_summary", {})
    matching_summary = final_report.get("matching_summary", {})
    skills_summary = final_report.get("skills_summary", {})
    semantic_analysis = final_report.get("semantic_analysis", {})
    recommendation_summary = final_report.get("recommendation_summary", {})

    markdown_report = f"""
# Final CV-Job Matching Report

## Candidate Information

- Candidate name: {candidate_information.get("candidate_name")}
- Email: {candidate_information.get("email")}
- Phone: {candidate_information.get("phone")}
- Location: {candidate_information.get("location")}

## Job Information

- Job title: {job_information.get("job_title")}
- Company: {job_information.get("company_name")}
- Job category: {job_information.get("job_category")}
- Location: {job_information.get("location")}
- Work mode: {job_information.get("work_mode")}
- Employment type: {job_information.get("employment_type")}

## CV Quality Summary

{cv_quality_summary.get("overall_summary")}

### CV Strengths

{create_markdown_list(cv_quality_summary.get("strengths", []), empty_message="- No strengths listed.")}

### CV Weaknesses

{create_markdown_list(cv_quality_summary.get("weaknesses", []), empty_message="- No weaknesses listed.")}

## Matching Summary

- Final hybrid score: {matching_summary.get("final_hybrid_score")}/100
- Rule-based score: {matching_summary.get("rule_based_score")}/100
- LLM semantic score: {matching_summary.get("semantic_score")}/100
- Match category: {matching_summary.get("match_category")}

## Matched Required Skills

{create_matched_items_list(skills_summary.get("matched_required_skills", []), empty_message="- No matched required skills.")}

## Missing Required Skills

{create_markdown_list(skills_summary.get("missing_required_skills", []), empty_message="- No missing required skills.")}

## Matched Technology Skills

{create_matched_items_list(skills_summary.get("matched_technology_skills", []), empty_message="- No matched technology skills.")}

## Missing Technology Skills

{create_markdown_list(skills_summary.get("missing_technology_skills", []), empty_message="- No missing technology skills.")}

## LLM Semantic Analysis

### Role Fit Summary

{semantic_analysis.get("role_fit_summary")}

### Responsibilities Not Clearly Evidenced

{create_markdown_list(semantic_analysis.get("responsibilities_not_evidenced", []), empty_message="- No responsibility gaps listed.")}

### Soft Skills Not Clearly Evidenced

{create_markdown_list(semantic_analysis.get("soft_skills_not_clearly_evidenced", []), empty_message="- No soft skill evidence gaps listed.")}

## Recommendation Summary

{recommendation_summary.get("overall_recommendation_summary")}

## Priority Actions

{create_markdown_list(
    [item.get("action") for item in recommendation_summary.get("priority_actions", []) if isinstance(item, dict)],
    empty_message="- No priority actions listed."
)}

## Evidence-Based Notes

{create_markdown_list(recommendation_summary.get("evidence_based_notes", []), empty_message="- No evidence-based notes listed.")}

## Final Note

This report was generated through the LangGraph workflow.
"""

    return markdown_report

## 5. Define Workflow Nodes

### 5.1 Initialize Workflow Node

In [ ]:
def initialize_workflow_node(state: CVJobMatchingWorkflowState):
    started_at = datetime.now().isoformat(timespec="seconds")

    return {
        "started_at": started_at,
        "status": "running",
        "errors": [],
        "workflow_log": [
            f"{started_at} | Workflow started."
        ]
    }

### 5.2 Check Required Files Node

In [ ]:
def check_required_files_node(state: CVJobMatchingWorkflowState):
    required_paths = [
    state["cv_pdf_path"],
    state["cv_quality_path"],
    state["structured_cv_path"],
    state["structured_job_path"],
    state["matching_result_path"],
    state["recommendations_path"]
    ]   

    missing_files = []

    for path in required_paths:
        file_path = Path(path)

        if not file_path.exists():
            missing_files.append(str(file_path))

    if missing_files:
        error_message = f"Missing required files: {missing_files}"
        raise FileNotFoundError(error_message)

    return {
        "workflow_log": append_workflow_log(
            state,
            "All required input files are available."
        )
    }

### 5.3 Load Extracted CV Text Node

In [ ]:
def extract_cv_text_from_pdf_node(state: CVJobMatchingWorkflowState):
    cv_pdf_path = Path(state["cv_pdf_path"])
    cv_text_output_path = Path(state["cv_text_output_path"])

    cv_text_output_path.parent.mkdir(parents=True, exist_ok=True)

    cv_text = extract_text_from_pdf(cv_pdf_path)

    if not cv_text:
        raise ValueError("No text could be extracted from the CV PDF.")

    with open(cv_text_output_path, "w", encoding="utf-8") as file:
        file.write(cv_text)

    return {
        "cv_text": cv_text,
        "workflow_log": append_workflow_log(
            state,
            "CV text extracted from PDF successfully."
        )
    }

### 5.4 Load CV Quality Analysis Node

In [ ]:
def load_cv_quality_analysis_node(state: CVJobMatchingWorkflowState):
    cv_quality_analysis = load_json_file(state["cv_quality_path"])

    return {
        "cv_quality_analysis": cv_quality_analysis,
        "workflow_log": append_workflow_log(
            state,
            "CV quality analysis loaded successfully."
        )
    }

### 5.5 Load Structured CV Node

In [ ]:
def load_structured_cv_node(state: CVJobMatchingWorkflowState):
    structured_cv = load_json_file(state["structured_cv_path"])

    return {
        "structured_cv": structured_cv,
        "workflow_log": append_workflow_log(
            state,
            "Structured CV loaded successfully."
        )
    }

### 5.6 Load Structured Job Posting Node

In [ ]:
def load_structured_job_node(state: CVJobMatchingWorkflowState):
    structured_job = load_json_file(state["structured_job_path"])

    return {
        "structured_job": structured_job,
        "workflow_log": append_workflow_log(
            state,
            "Structured job posting loaded successfully."
        )
    }

### 5.7 Load Matching Result Node

In [ ]:
def load_matching_result_node(state: CVJobMatchingWorkflowState):
    matching_result = load_json_file(state["matching_result_path"])

    return {
        "matching_result": matching_result,
        "workflow_log": append_workflow_log(
            state,
            "Hybrid matching result loaded successfully."
        )
    }

### 5.8 Load Recommendations Node

In [ ]:
def load_recommendations_node(state: CVJobMatchingWorkflowState):
    recommendations_output = load_json_file(state["recommendations_path"])

    return {
        "recommendations_output": recommendations_output,
        "workflow_log": append_workflow_log(
            state,
            "Recommendations loaded successfully."
        )
    }

### 5.9 Final Report Generation Node

In [ ]:
def final_report_generation_node(state: CVJobMatchingWorkflowState):
    final_report_json = Path(state["final_report_json_path"])
    final_report_markdown = Path(state["final_report_markdown_path"])

    if final_report_json.exists():
        final_report = load_json_file(final_report_json)

        if final_report_markdown.exists():
            with open(final_report_markdown, "r", encoding="utf-8") as file:
                final_markdown_report = file.read()
        else:
            final_markdown_report = ""

        log_message = "Final report loaded from notebook 10 output."

    else:
        final_report_json.parent.mkdir(parents=True, exist_ok=True)

        final_report = create_compact_final_report_from_state(state)
        final_markdown_report = create_compact_markdown_report(final_report)

        save_json_file(final_report, final_report_json)

        with open(final_report_markdown, "w", encoding="utf-8") as file:
            file.write(final_markdown_report)

        log_message = "Notebook 10 final report was not found. Compact final report generated inside LangGraph workflow."

    return {
        "final_report": final_report,
        "final_markdown_report": final_markdown_report,
        "workflow_log": append_workflow_log(
            state,
            log_message
        )
    }

### 5.10 Finish Workflow Node

In [ ]:
def finish_workflow_node(state: CVJobMatchingWorkflowState):
    completed_at = datetime.now().isoformat(timespec="seconds")

    return {
        "completed_at": completed_at,
        "status": "completed",
        "workflow_log": append_workflow_log(
            state,
            "Workflow completed successfully."
        )
    }

### 5.11 Save Workflow Output Node

In [ ]:
def save_workflow_output_node(state: CVJobMatchingWorkflowState):
    matching_result = state.get("matching_result", {})
    final_result = matching_result.get("final_result", {})

    structured_cv = state.get("structured_cv", {})
    structured_job = state.get("structured_job", {})

    workflow_output = {
        "metadata": {
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "workflow_type": "langgraph_cv_job_matching_workflow",
            "status": state.get("status"),
            "started_at": state.get("started_at"),
            "completed_at": state.get("completed_at")
        },
        "input_files": {
            "cv_text": state.get("cv_text_path"),
            "cv_quality": state.get("cv_quality_path"),
            "structured_cv": state.get("structured_cv_path"),
            "structured_job": state.get("structured_job_path"),
            "matching_result": state.get("matching_result_path"),
            "recommendations": state.get("recommendations_path")
        },
        "output_files": {
            "final_report_json": state.get("final_report_json_path"),
            "final_report_markdown": state.get("final_report_markdown_path"),
            "workflow_result": str(workflow_result_path),
            "workflow_log": str(workflow_log_path)
        },
        "candidate_information": {
            "candidate_name": structured_cv.get("candidate_name"),
            "location": structured_cv.get("location")
        },
        "job_information": {
            "job_title": structured_job.get("job_title"),
            "company_name": structured_job.get("company_name"),
            "job_category": structured_job.get("job_category")
        },
        "matching_summary": {
            "final_hybrid_score": final_result.get("final_hybrid_score"),
            "rule_based_score": final_result.get("rule_based_score"),
            "semantic_score": final_result.get("semantic_score"),
            "match_category": final_result.get("match_category")
        },
        "workflow_log": state.get("workflow_log", [])
    }

    save_json_file(workflow_output, workflow_result_path)

    workflow_log_markdown = "# LangGraph Workflow Log\n\n"
    workflow_log_markdown += "\n".join([f"- {item}" for item in state.get("workflow_log", [])])

    with open(workflow_log_path, "w", encoding="utf-8") as file:
        file.write(workflow_log_markdown)

    return {
        "workflow_output": workflow_output,
        "workflow_log": append_workflow_log(
            state,
            "Workflow execution summary saved successfully."
        )
    }

## 6. Build LangGraph Workflow

In [111]:
workflow = StateGraph(CVJobMatchingWorkflowState)

In [ ]:
workflow.add_node("initialize_workflow", initialize_workflow_node)
workflow.add_node("check_required_files", check_required_files_node)
workflow.add_node("extract_cv_text_from_pdf", extract_cv_text_from_pdf_node)
workflow.add_node("load_cv_quality_analysis", load_cv_quality_analysis_node)
workflow.add_node("load_structured_cv", load_structured_cv_node)
workflow.add_node("load_structured_job", load_structured_job_node)
workflow.add_node("load_matching_result", load_matching_result_node)
workflow.add_node("load_recommendations", load_recommendations_node)
workflow.add_node("final_report_generation", final_report_generation_node)
workflow.add_node("finish_workflow", finish_workflow_node)
workflow.add_node("save_workflow_output", save_workflow_output_node)

In [ ]:
workflow.add_edge(START, "initialize_workflow")
workflow.add_edge("initialize_workflow", "check_required_files")
workflow.add_edge("check_required_files", "extract_cv_text_from_pdf")
workflow.add_edge("extract_cv_text_from_pdf", "load_cv_quality_analysis")
workflow.add_edge("load_cv_quality_analysis", "load_structured_cv")
workflow.add_edge("load_structured_cv", "load_structured_job")
workflow.add_edge("load_structured_job", "load_matching_result")
workflow.add_edge("load_matching_result", "load_recommendations")
workflow.add_edge("load_recommendations", "final_report_generation")
workflow.add_edge("final_report_generation", "finish_workflow")
workflow.add_edge("finish_workflow", "save_workflow_output")
workflow.add_edge("save_workflow_output", END)

In [113]:
compiled_workflow = workflow.compile()

## 8. Run LangGraph Workflow

In [114]:
initial_state = {
    "cv_pdf_path": str(CV_PDF_PATH),
    "cv_text_output_path": str(CV_TEXT_OUTPUT_PATH),
    "cv_quality_path": str(cv_quality_path),
    "structured_cv_path": str(structured_cv_path),
    "structured_job_path": str(structured_job_path),
    "matching_result_path": str(matching_result_path),
    "recommendations_path": str(recommendations_path),
    "final_report_json_path": str(final_report_json_path),
    "final_report_markdown_path": str(final_report_markdown_path)
}

In [119]:
final_state = compiled_workflow.invoke(initial_state)

In [120]:
print("Workflow status:", final_state.get("status"))

Workflow status: completed


## 9. Preview

In [121]:
for log_item in final_state.get("workflow_log", []):
    print(log_item)

2026-07-20T19:14:05 | Workflow started.
2026-07-20T19:14:05 | All required input files are available.
2026-07-20T19:14:05 | CV text extracted from PDF successfully.
2026-07-20T19:14:05 | CV quality analysis loaded successfully.
2026-07-20T19:14:05 | Structured CV loaded successfully.
2026-07-20T19:14:05 | Structured job posting loaded successfully.
2026-07-20T19:14:05 | Hybrid matching result loaded successfully.
2026-07-20T19:14:05 | Recommendations loaded successfully.
2026-07-20T19:14:05 | Final report loaded from notebook 10 output.
2026-07-20T19:14:05 | Workflow completed successfully.
2026-07-20T19:14:05 | Workflow execution summary saved successfully.


In [122]:
final_state.get("workflow_output", {})

{'metadata': {'created_at': '2026-07-20T19:14:05',
  'workflow_type': 'langgraph_cv_job_matching_workflow',
  'status': 'completed',
  'started_at': '2026-07-20T19:14:05',
  'completed_at': '2026-07-20T19:14:05'},
 'input_files': {'cv_text': None,
  'cv_quality': '..\\outputs\\cv_quality\\cv_quality_analysis.json',
  'structured_cv': '..\\outputs\\cv_extraction\\structured_cv.json',
  'structured_job': '..\\outputs\\job_extraction\\structured_job.json',
  'matching_result': '..\\outputs\\matching\\matching_result.json',
  'recommendations': '..\\outputs\\recommendations\\recommendations.json'},
 'output_files': {'final_report_json': '..\\outputs\\final_report\\final_report.json',
  'final_report_markdown': '..\\outputs\\final_report\\final_report.md',
  'workflow_result': '..\\outputs\\langgraph_workflow\\workflow_result.json',
  'workflow_log': '..\\outputs\\langgraph_workflow\\workflow_log.md'},
 'candidate_information': {'candidate_name': 'ALEX JOHNSON',
  'location': 'Example City,

In [123]:
print(final_state.get("final_markdown_report", "Final Markdown report is not available."))


# Final CV-Job Matching Report

## 1. Candidate Information

- Candidate name: ALEX JOHNSON
- Email: alex.jjohnson@example.com
- Phone: +1 202-555-0147
- Location: Example City, 10000
- LinkedIn: None
- GitHub: None
- Portfolio: None

## 2. Job Information

- Job title: Senior Developer
- Company: USLI
- Job category: Software Development
- Location: None
- Work mode: On-site 75% of the time
- Employment type: Full-time

## 3. CV Quality Summary

- Final CV quality score: 70/100
- CV quality category: Good CV

### Overall CV Quality Summary

The CV presents a strong candidate profile with relevant education and project experience in software engineering. However, it lacks clarity in the organization of skills and measurable results, which could enhance its impact.

### CV Quality Score Breakdown

- Structure and readability: 70/100
- Completeness: 80/100
- Technical skills clarity: 60/100
- Experience description: 75/100
- Projects description: 80/100
- Measurable results: 40/100
- IT